# VisionAssist fine-tune v1 (Kaggle GPU)

Fine-tunes the existing ROD checkpoint with a COCO subset (and, once recorded, your own
phone footage) to improve real-world car/person detection while keeping all 25 ROD
classes and the ROD-specific hazards (stairs, manholes, poles, ...).

**This notebook does not change `models/rod_yolov8n_50ep_best.pt` or the app.** It writes
a new checkpoint under a new name; switching the app's default model is a separate,
later decision.

Pipeline: re-split ROD (no train/val/test overlap) -> build a COCO street-scene subset ->
complete missing labels in both -> **pause for a human label review** -> baseline
evaluation -> fine-tune (guarded, off by default) -> evaluate candidates against the
acceptance criteria in `docs/finetune_v1_plan.md`.

Attach as Kaggle inputs: the ROD dataset (`abtinzandi/obstacle-detection-dataset`) and a
COCO 2017 dataset (e.g. `awsaf49/coco-2017-dataset`). Enable a GPU (T4 or P100) and
Internet (to clone the repo).

## 1. Get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/SudhaMsr/VisionAssist-MVP.git"
BRANCH = "experiment/finetune-v1"  # change to test a different branch

if Path("/kaggle/working").exists():
    REPO_DIR = Path("/kaggle/working/VisionAssist-MVP")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    WORK = Path("/kaggle/working/ft_v1")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    WORK = REPO_DIR / "outputs" / "ft_v1"

assert (REPO_DIR / "training").is_dir(), f"training/ package not found under {REPO_DIR}"
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
WORK.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_DIR)
print("Working directory for the built dataset:", WORK)

## 2. Dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
print("ultralytics installed; CUDA available:", torch.cuda.is_available())

## 3. Locate the datasets

Set these to the Kaggle input paths for your attached datasets. `PHONE_ROOT` is optional:
leave it `None` until you have recorded and labelled the phone walks
(`docs/finetune_v1_plan.md` says exactly what to record; `training/phone_frames.py`
extracts frames for labelling). Without it, run A trains on ROD + COCO only, and the
real-world acceptance checks in Step 8 are skipped with a clear "MISSING" message
rather than silently passing.

In [ ]:
ROD_ROOT = Path("/kaggle/input/obstacle-detection-dataset/ROD-Dataset/dataset")
COCO_ROOT = Path("/kaggle/input")  # searched recursively for coco2017/annotations/instances_*.json
PHONE_ROOT = None  # e.g. Path("/kaggle/input/visionassist-phone-walks") once recorded and labelled

for name, path, required in (("ROD_ROOT", ROD_ROOT, True), ("COCO_ROOT", COCO_ROOT, True),
                              ("PHONE_ROOT", PHONE_ROOT, False)):
    if path is None:
        print(f"{name}: not set (optional, skipped)")
    elif path.exists():
        print(f"{name}: {path}")
    elif required:
        raise FileNotFoundError(f"{name} not found at {path}. Attach the dataset in Kaggle's Input panel.")
    else:
        raise FileNotFoundError(f"{name} was set but does not exist: {path}")

## 4. Re-split ROD (no train/val/test overlap)

About 21% of ROD's own validation/test images share a source photo or video with
training (Roboflow duplicates and video frames), which makes today's reported scores
optimistic. `configs/finetune_v1/splits/` already has the committed re-split (grouped by
source, seed 42); this cell only verifies it against the attached dataset and regenerates
it if the dataset changed. `prepare-rod` then materialises the three splits under `WORK`
(images are symlinks; the attached dataset is never modified).

In [ ]:
from training.splits import main as splits_main
from training.build_dataset import main as build_main

SPLIT_DIR = REPO_DIR / "configs" / "finetune_v1" / "splits"
if not (SPLIT_DIR / "rod_train.txt").exists():
    print("No committed split found; generating one now (will be committed afterwards).")
    splits_main(["--dataset", str(ROD_ROOT)])
else:
    print("Using the committed split in", SPLIT_DIR)

build_main(["prepare-rod", "--rod", str(ROD_ROOT), "--work", str(WORK)])

## 5. Build the COCO street-scene subset

Keeps images with a car/bus/truck/motorcycle, or a person plus an outdoor cue. Skips
images containing a bicycle (ROD's "Bike" mixes bicycles and motorbikes and is left
unchanged in this experiment, so a bicycle could not be mapped consistently) and images
with unlabelled "crowd" regions. `--limit` keeps COCO from dominating training; raise it
if GPU time allows (the plan budgets about 10,000 train / 1,000 val images).

In [ ]:
from training.coco_subset import main as coco_main

coco_main(["--coco", str(COCO_ROOT), "--split", "train", "--limit", "10000", "--work", str(WORK)])
coco_main(["--coco", str(COCO_ROOT), "--split", "val", "--limit", "1000", "--work", str(WORK)])

## 6. Complete missing labels

ROD merges 26 sources that each labelled only their own classes, so many people/cars are
unlabelled; COCO never labels stairs, manholes, poles or the other ROD-only classes. This
adds high-confidence predictions as labels: the ROD model fills in ROD-only classes on the
COCO images, and the COCO model fills in people/vehicles on the ROD train/val images that
don't already have one. ROD's test split is never touched. `models/yolov8n.pt` (the COCO
assist model) is downloaded here if not already present; it is git-ignored.

In [ ]:
from ultralytics import YOLO
from training.label_completion import main as completion_main

if not Path("models/yolov8n.pt").is_file():
    YOLO("yolov8n.pt").save("models/yolov8n.pt")  # downloads the official COCO weights

completion_main(["--work", str(WORK), "--rod-model", "models/rod_yolov8n_50ep_best.pt",
                 "--coco-model", "models/yolov8n.pt", "--device", "0"])

## 7. STOP: review the added labels before training

`WORK/review_sheet_*.jpg` shows a random sample of boxes the step above added, with the
predicted class and confidence. Open a few sheets (`from IPython.display import Image;
Image(filename=str(WORK / "review_sheet_1.jpg"))`) and check that most are correct. If
more than about 10% look wrong, raise `--rod-conf` / `--coco-conf` in the previous cell
and re-run it. Do not continue to training until you have done this.

## 8. Assemble the dataset and evaluate the baselines

Writes `data_v1.yaml` (run A: ROD + COCO + any phone train/val walks) and
`data_v1_rod_only.yaml` (run B control: ROD only, same validation set), plus the held-out
test lists (`test_rod.txt`, `test_phone.txt`, `test_coco.txt`) that are never trained on.
Then evaluates **today's shipped model**, alone and with the COCO assist, on those test
sets — the baseline that any fine-tuned candidate must beat.

In [ ]:
build_main(["assemble", "--work", str(WORK)] + (["--phone", str(PHONE_ROOT)] if PHONE_ROOT else []))
print((WORK / "dataset_report.json").read_text())

In [ ]:
from training.evaluate import main as evaluate_main

BASELINE_OUT = WORK / "eval_baseline"
evaluate_main([
    "--work", str(WORK), "--device", "0", "--out", str(BASELINE_OUT),
    "--system", "today=models/rod_yolov8n_50ep_best.pt",
    "--system", "today+assist=models/rod_yolov8n_50ep_best.pt+models/yolov8n.pt",
])

## 9. Fine-tune (guarded — set `RUN_TRAINING = True` only after Steps 7-8 look right)

Both runs start from the shipped checkpoint (never from scratch) with the lower learning
rate and settings pinned in `configs/finetune_v1/train.yaml`. Run A is the candidate
(ROD + COCO [+ phone]); run B is a same-schedule control on ROD alone, to separate "the
new data helped" from "more training helped". Each takes roughly 1.5-2 hours on a T4.

In [ ]:
RUN_TRAINING = False  # <-- set to True to actually start fine-tuning

import yaml
from ultralytics import YOLO

CFG = yaml.safe_load((REPO_DIR / "configs" / "finetune_v1" / "train.yaml").read_text())
RUNS_DIR = WORK / "runs"
results = {}

if RUN_TRAINING:
    for run_id, run_cfg in CFG["runs"].items():
        model = YOLO(CFG["start_checkpoint"])
        model.train(data=str(WORK / run_cfg["data"]), project=str(RUNS_DIR), name=run_cfg["name"],
                    exist_ok=True, **{k: v for k, v in CFG["common"].items()})
        results[run_id] = RUNS_DIR / run_cfg["name"] / "weights" / "best.pt"
        print(run_id, "->", results[run_id])
else:
    print("RUN_TRAINING is False: skipping fine-tuning. Review Steps 7-8 first.")

## 10. Evaluate the candidates against the acceptance criteria

Compares the baselines with runs A and B on the same held-out sets, and checks the
pass/fail criteria from `docs/finetune_v1_plan.md` (recall gain on phone footage, no ROD
class regressing, fall hazards never pushed out of the alert slots). Only runs if
training above actually ran.

In [ ]:
if RUN_TRAINING:
    FINAL_OUT = WORK / "eval_final"
    systems = ["today=models/rod_yolov8n_50ep_best.pt",
               "today+assist=models/rod_yolov8n_50ep_best.pt+models/yolov8n.pt",
               f"A={results['A']}", f"B={results['B']}"]
    evaluate_main(["--work", str(WORK), "--device", "0", "--out", str(FINAL_OUT),
                  "--assist-baseline", "today+assist", "--candidate", "A", "--candidate", "B",
                  *[f"--system={s}" for s in systems]])
else:
    print("Nothing to evaluate: training did not run.")

## 11. Bring the results back (see `docs/finetune_v1_plan.md` Git workflow)

Download this notebook's Output (the `ft_v1/` folder: metrics, review sheets, split
reports, and `runs/*/weights/best.pt` if training ran) from the Kaggle website — there are
no Kaggle API credentials configured locally yet. Locally: verify a checksum, copy the
chosen checkpoint to `models/rod_coco_ft_v1_best.pt` (a new file; the shipped model is
never overwritten), copy the metrics into `results/rod_coco_ft_v1/`, and open a PR from
this branch with the comparison table. Only a separate, later PR — with your explicit
approval — would change the app's default model.